### Year 4 Topology 5 Contingency Analysis Reporting - Upper Limit Voltage Violation

In [1]:
import pandas as pd
import numpy as np

#### Bus Voltage Data

In [2]:
list_volt = []
list_gens = [0, 50, 100]
list_lsc = ['LLS','RLS','HLS']
list_gen_hydro = [400,500,600] #dry, average, wet hydological scenarios
list_gen_wind = [0,40,75] # for 3 wind gust scenarios
for gen in list_gens:
    for gen_hy in list_gen_hydro:
        for lsc in list_lsc:
            for gen_wi in list_gen_wind:
                file_out = 'savnw_sol_' + str(gen) +'_hy_' +str(gen_hy) +'_wi_' + str(gen_wi) +'_' + str(lsc)+'.xlsx'
                data_volt = pd.read_excel(file_out, sheet_name='Bus Voltage', usecols = ['BUS', 'RECORD', 'TYPE', 'MIN/DROP', 'MAX/RISE', 'CONTINGENCY',
                                        'BASE VOLTS', 'CONT VOLTS', 'DEVIATION', 'RANGE VIO', 'DEV VIO'] )
                data_vo = data_volt.dropna(how='all').reset_index(drop=True)
                data_vo['Scenario']= 'Solar = ' + str(gen) + ' MW, ' + 'Hydro = ' + str(gen_hy) + ' MW, ' + 'Wind = ' + str(gen_wi) + ' MW, '  + lsc.upper()
                list_volt.append(data_vo)
data_v = pd.concat(list_volt).reset_index(drop=True)

#### Bus voltage data wrangling 

In [3]:
data_v = data_v.rename(columns={'BUS':'Bus', 
                                'CONTINGENCY':'Contingency',
                                'BASE VOLTS':'Base Voltage',                               
                                'CONT VOLTS':'Contingency Voltage',
                                'RANGE VIO' : 'Range Violation', 
                                'DEVIATION' : 'Deviation'})
data_v['Contingency'] = data_v['Contingency'].str.replace('&','\&')
data_v['Bus'] = data_v['Bus'].str.replace('_','\_')
data_v['Bus Number'] = data_v['Bus'].str.split(expand=True)[0]

#### Upper limit bus voltage violations

In [4]:
data_high = data_v[data_v['Contingency Voltage']>1.1].reset_index(drop=True)
print(len(data_high))
display(data_high.head())

288


,Bus,RECORD,TYPE,MIN/DROP,MAX/RISE,Contingency,Base Voltage,Contingency Voltage,Deviation,Range Violation,DEV VIO,Scenario,Bus Number
0,211 HYDRO\_G 20.000,ALL,RANGE,0.95,1.05,SING OPN LIN 10 201-202(1),1.046177,1.103840,0.057664,0.053840,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, HLS",211
1,212 HYDRO\_N 20.000,ALL,RANGE,0.95,1.05,SING OPN LIN 10 201-202(1),1.046177,1.103840,0.057664,0.053840,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, HLS",212
2,211 HYDRO\_G 20.000,LIMIT,RANGE,0.90,1.10,SING OPN LIN 10 201-202(1),1.046177,1.103840,0.057664,0.003840,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, HLS",211
3,212 HYDRO\_N 20.000,LIMIT,RANGE,0.90,1.10,SING OPN LIN 10 201-202(1),1.046177,1.103840,0.057664,0.003840,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, HLS",212
4,211 HYDRO\_G 20.000,ALL,RANGE,0.95,1.05,BUS 3005,1.046177,1.105477,0.059300,0.055477,NaN,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, HLS",211


#### Reporting of Buses with upper voltage violations

In [5]:
H_scen = list(data_high['Scenario'].unique())
all_scen = list(data_v['Scenario'].unique())
list_diff = ', '.join(list(set(all_scen) - set(H_scen)))
print(f'It was seen that for Topology 3, {list_diff} did not report any upper voltage limit violation(s).')
list_viol = ', '.join(H_scen)
print(f'For Topology 1, the scenarios, {list_viol} reported upper voltage limit violation(s).')
print('------------------------------------------------------------------------------------------------------------------------------------------------------')
for vhscen in H_scen:
    # Extracting bus data with voltage more than 1.1 PU
    print(vhscen)
    data_hscen = data_high[data_high['Scenario'] == vhscen]
    print(f'For the studied scenario {vhscen}, the buses violating the Upper voltage limits are bus(es)',', '.join(data_hscen['Bus Number'].unique()))
    
    # upper range violation latex table data reporting
    data_rephv = data_hscen[['Bus Number','Contingency','Base Voltage', 'Contingency Voltage', 'Deviation', 'Range Violation']].reset_index(drop=True)
    
    unit_hiv = data_rephv[data_rephv['Contingency'].str.contains('UNIT')]
    if len(unit_hiv) != 0:
        unit_hv = unit_hiv.drop_duplicates(subset=['Bus Number'])
        for conthu in list(unit_hv['Contingency'].unique()):
            unit_cont = unit_hv[unit_hv['Contingency']==conthu]
            busesu = ', '.join(unit_cont['Bus Number'].unique())
            print(f'The buses {busesu} reported violation for the unit fault {conthu}')
            
    
    bus_hiv = data_rephv[data_rephv['Contingency'].str.contains('BUS')]
    if len(bus_hiv) != 0:
        bus_hv = bus_hiv.drop_duplicates(subset=['Bus Number'])
        for conthb in list(bus_hv['Contingency'].unique()):
            bus_cont = bus_hv[bus_hv['Contingency']==conthb]
            busesb = ', '.join(bus_cont['Bus Number'].unique())
            print(f'The buses {busesb} reported violation for the bus fault {conthb}')
            
        
    line_hiv = data_rephv[data_rephv['Contingency'].str.contains('SING OPN LIN')]
    if len(line_hiv) != 0:
        line_hv = line_hiv.drop_duplicates(subset=['Bus Number'])
        for conthl in list(line_hv['Contingency'].unique()):
            line_cont = line_hv[line_hv['Contingency']==conthl]
            busesl = ', '.join(line_cont['Bus Number'].unique())
            print(f'The buses {busesl} reported violation for the single line open fault {conthl}')
    print('------------------------------------------------------------------------------------------------------------------------------------------------------')  

It was seen that for Topology 3, Solar = 0 MW, Hydro = 600 MW, Wind = 75 MW, LLS, Solar = 0 MW, Hydro = 600 MW, Wind = 0 MW, LLS, Solar = 100 MW, Hydro = 400 MW, Wind = 40 MW, RLS, Solar = 0 MW, Hydro = 400 MW, Wind = 40 MW, LLS, Solar = 0 MW, Hydro = 400 MW, Wind = 75 MW, RLS, Solar = 50 MW, Hydro = 500 MW, Wind = 0 MW, RLS, Solar = 50 MW, Hydro = 500 MW, Wind = 40 MW, LLS, Solar = 50 MW, Hydro = 600 MW, Wind = 40 MW, LLS, Solar = 50 MW, Hydro = 500 MW, Wind = 40 MW, RLS, Solar = 50 MW, Hydro = 400 MW, Wind = 0 MW, HLS, Solar = 50 MW, Hydro = 400 MW, Wind = 0 MW, RLS, Solar = 50 MW, Hydro = 500 MW, Wind = 0 MW, LLS, Solar = 50 MW, Hydro = 600 MW, Wind = 75 MW, LLS, Solar = 0 MW, Hydro = 500 MW, Wind = 75 MW, LLS, Solar = 100 MW, Hydro = 600 MW, Wind = 75 MW, LLS, Solar = 100 MW, Hydro = 600 MW, Wind = 0 MW, LLS, Solar = 50 MW, Hydro = 600 MW, Wind = 0 MW, LLS, Solar = 50 MW, Hydro = 400 MW, Wind = 75 MW, RLS, Solar = 100 MW, Hydro = 600 MW, Wind = 40 MW, LLS, Solar = 0 MW, Hydro = 600

Upper Voltage Limit Counts - Scenario 

In [6]:
vh_index = list(data_high['Scenario'].value_counts().index)
vh_counts = list(data_high['Scenario'].value_counts())
dict_vh_count = {
    'Scenario':vh_index,
    'Violation Counts':vh_counts
}

scen_hv_vc  = pd.DataFrame(dict_vh_count)
scen_hv_vc

,Scenario,Violation Counts
0,"Solar = 100 MW, Hydro = 600 MW, Wind = 75 MW, HLS",16
1,"Solar = 100 MW, Hydro = 600 MW, Wind = 40 MW, HLS",16
2,"Solar = 0 MW, Hydro = 600 MW, Wind = 0 MW, RLS",16
3,"Solar = 0 MW, Hydro = 500 MW, Wind = 0 MW, HLS",12
4,"Solar = 0 MW, Hydro = 500 MW, Wind = 40 MW, HLS",12
5,"Solar = 0 MW, Hydro = 500 MW, Wind = 75 MW, HLS",12
6,"Solar = 0 MW, Hydro = 600 MW, Wind = 40 MW, RLS",12
7,"Solar = 0 MW, Hydro = 600 MW, Wind = 75 MW, RLS",12
8,"Solar = 100 MW, Hydro = 600 MW, Wind = 0 MW, HLS",12
9,"Solar = 0 MW, Hydro = 400 MW, Wind = 40 MW, HLS",8


Upper Voltage Limit Counts - Bus

In [7]:
vhb_index = list([i.strip().split()[0] for i in data_high['Bus'].value_counts().index])
vhb_counts = list(data_high['Bus'].value_counts())
dict_vhb_count = {
    'Bus':vhb_index,
    'Violation Counts':vhb_counts
}

bus_hv_vc  = pd.DataFrame(dict_vhb_count)
bus_hv_vc

,Bus,Violation Counts
0,211,144
1,212,144


Upper Voltage Limit Counts - Contingency

In [8]:
vh_index = list(data_high['Contingency'].value_counts().index)
vh_counts = list(data_high['Contingency'].value_counts())
dict_vh_count = {
    'Contingency':vh_index,
    'Violation Counts':vh_counts
}

scen_hv_vc  = pd.DataFrame(dict_vh_count)
scen_hv_vc

,Contingency,Violation Counts
0,BUS 3005,56
1,SING OPN LIN 20 205-206(1),40
2,UNIT 206(1),40
3,SING OPN LIN 4 151-152(1),36
4,SING OPN LIN 5 151-152(2),36
5,SING OPN LIN 6 152-153(1),24
6,BUS 203,24
7,SING OPN LIN 10 201-202(1),20
8,BUS 3004,12


Upper Voltage Result Summary - grouped by Scenario and Contingency

In [9]:
data_fil_hv = data_high[['Scenario', 'Bus Number', 'Contingency']].drop_duplicates()
pivot_hv = pd.DataFrame(pd.pivot(data_fil_hv, index= ['Scenario','Contingency'], columns = 'Bus Number',values = 'Bus Number').to_records())
pivot_hv['Buses'] = pivot_hv['211'].astype(str).str.cat(pivot_hv[['212']].astype(str), sep=',')
pivot_hv['Buses'] =pivot_hv['Buses'].str.replace(',nan','').str.replace('nan,','')
pivot_high = pivot_hv.drop(columns = list(data_fil_hv['Bus Number'].unique()))

In [10]:
pivot_table_hv = pd.DataFrame(pd.pivot_table(data_fil_hv, index= ['Scenario','Contingency'],values = 'Bus Number', aggfunc='count').to_records())
pivot_table_high = pivot_table_hv.rename(columns = {'Bus Number':'Bus Count'})
df_high = pd.merge(pivot_high, pivot_table_high, on=['Scenario','Contingency'], how='inner').sort_values(by='Bus Count', ascending=False).reset_index(drop=True)
df_high

,Scenario,Contingency,Buses,Bus Count
0,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, HLS",BUS 3005,"211,212",2
1,"Solar = 0 MW, Hydro = 400 MW, Wind = 0 MW, HLS",SING OPN LIN 10 201-202(1),"211,212",2
2,"Solar = 100 MW, Hydro = 600 MW, Wind = 75 MW, HLS",SING OPN LIN 4 151-152(1),"211,212",2
3,"Solar = 100 MW, Hydro = 600 MW, Wind = 75 MW, HLS",BUS 203,"211,212",2
4,"Solar = 100 MW, Hydro = 600 MW, Wind = 40 MW, RLS",SING OPN LIN 10 201-202(1),"211,212",2
...,...,...,...,...
67,"Solar = 0 MW, Hydro = 600 MW, Wind = 75 MW, HLS",BUS 3004,"211,212",2
68,"Solar = 0 MW, Hydro = 600 MW, Wind = 40 MW, RLS",SING OPN LIN 5 151-152(2),"211,212",2
69,"Solar = 0 MW, Hydro = 600 MW, Wind = 40 MW, RLS",SING OPN LIN 4 151-152(1),"211,212",2
70,"Solar = 0 MW, Hydro = 600 MW, Wind = 40 MW, RLS",BUS 3005,"211,212",2
